In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185,

In [ ]:
import importlib
from sklearn.model_selection import StratifiedKFold, KFold
import cup_common as cc
import knn_common as kc
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor


In [ ]:
importlib.reload(cc)
importlib.reload(kc)

In [ ]:
# Split for KFold
n_split = 5
default_cv = KFold(n_splits=n_split, shuffle=True, random_state=42)
default_krange = list(range(1, 41, 2))
default_scoring = "neg_mean_absolute_error"

default_pipe = Pipeline([
    ("knn", KNeighborsRegressor())
])

default_param_grid = {
    "knn__n_neighbors": default_krange,
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],          # 1=Manhattan, 2=Euclidean
    "knn__metric": ["minkowski"]
}




<h2>Cup Dataset</h2>
<hr/>

<h4>Data Loading</h4>
<p>Load and introspect data from monk training and test set.</p>

In [ ]:
df_train, df_test = cc.load_set()
X_tr, y_tr, X_ts, y_ts = cc.split_and_prepare_dataset(df_train, ratio=0.2)
features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

In [ ]:
gs = GridSearchCV(
    default_pipe,
    param_grid=default_param_grid,
    scoring=default_scoring,
    cv=default_cv,
    n_jobs=-1,
    return_train_score=True
)

best_estimator = gs.fit(X_tr, y_tr)
model, params = kc.extract_best_knn_metrics_from_grid(gs)

print(params)

In [ ]:
kc.plot_knn_validation_curve_from_gs(gs,"mean")

In [ ]:
#y_tr_np = y_tr.to_numpy()
kc.plot_knn_learning_curve(model, X_tr, y_tr, default_cv, scoring=default_scoring)

In [ ]:
importlib.reload(kc)
kc.plot_knn_learning_curves_grid(
    model,
    X=X_tr,
    y=y_tr,
    cv=default_cv,
    scoring=default_scoring
)

<h4>Prediction</h4>
<p>Performing a prediction on the test set obtained from train set</p>

In [ ]:
# Model has already been fitted by gridsearch
y_pred = model.predict(X_ts)

In [ ]:
report = kc.regression_report(y_ts, y_pred, name="KNN Regressor")